# SHIPIT Agent: Session History And Seeded Chat History

This notebook shows both ways to preserve context:

- seed `history=[Message(...)]`
- use `session_store` and `session_id` for persistent chat history

In [1]:
from pathlib import Path
import sys
from IPython.display import Markdown, display

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.run_multi_tool_agent import build_llm_from_env
from shipit_agent import Agent, FileSessionStore, Message

In [3]:
llm = build_llm_from_env("bedrock")
seeded_agent = Agent(
    llm=llm,
    history=[
        Message(role="user", content="We are building an internal project assistant."),
        Message(
            role="assistant",
            content="Understood. I will keep answers focused on developer workflows.",
        ),
    ],
)

seeded_result = seeded_agent.run("What prior context do you have?")
display(Markdown(f"**Seeded Agent Response:** {seeded_result.output}"))

**Seeded Agent Response:** I don’t retain any memory between separate conversations, so the only context I have right now is what’s been exchanged in this session:

1. **You** mentioned that you’re building an internal project assistant.  
2. **You** asked what prior context I have.

Beyond that, I have no additional background (e.g., your organization’s architecture, the technologies you’re using, specific requirements, or any previous discussions). If you’d like me to take previous design notes, constraints, or goals into account, just paste or summarize them here and I’ll incorporate them into my responses.

In [4]:
session_store = FileSessionStore(ROOT / ".shipit_notebooks" / "sessions_demo")

session_agent = Agent(
    llm=llm,
    session_store=session_store,
    session_id="demo-chat",
)

session_agent.run("My project codename is Atlas.")
follow_up = session_agent.run("What project codename did I mention earlier?")
display(Markdown(f"**Follow-up Response:** {follow_up.output}"))

**Follow-up Response:** You said the project codename is **Atlas**.

In [5]:
chat_session = session_agent.chat_session(session_id="demo-chat")
chat_session.history()

[Message(role='system', content="You are Shipit, a capable general-purpose agent runtime.\n\nCore behavior:\n- Be accurate, direct, and execution-oriented.\n- Solve the user's task end-to-end when possible instead of stopping at analysis.\n- Use tools when they materially improve correctness, freshness, or efficiency.\n- Prefer structured evidence over guesses.\n\nTool behavior:\n- Read tool descriptions and tool prompts carefully before calling them.\n- Use the smallest correct tool for the job.\n- When a task is complex, plan before acting.\n- When information may be outdated, prefer web and external tools over stale assumptions.\n- When a task needs files, artifacts, or code execution, use the relevant tools instead of simulating output.\n\nQuality bar:\n- Keep outputs clear and complete.\n- Verify important results before returning them.\n- Surface residual uncertainty instead of hiding it.\n- Avoid repeated failed actions; adjust strategy after an error.", name=None, metadata={'ag